# Part 0 — Data Preparation

**Goal of this notebook:** load the raw multi-omics TCGA-BRCA data, split patients into
train/test sets, and save that split to disk so every later notebook in the workshop
uses the *exact same* patients for training and testing.

We work with three "omic views" for each patient:

| View              | What it measures                          | # features |
|--------------------|--------------------------------------------|-----------:|
| `transcriptomics`  | gene expression (RNA-seq)                  | ~30,000    |
| `proteomics`        | protein abundance                          | ~460       |
| `methylation`       | DNA methylation (epigenetic marks)         | ~200,000   |

The prediction target is the **PAM50 molecular subtype** of breast cancer
(`LumA`, `LumB`, `Basal`, `Her2`, `Normal`).

## 1. Imports

In [2]:
from pathlib import Path
import pickle

from helpers import load_omics
from sklearn.model_selection import train_test_split

ModuleNotFoundError: No module named 'sklearn'

## 2. Load the multi-omics data

`load_omics` reads the pre-processed TCGA-BRCA data for the requested omic views and
returns:

- `X_views` — a dict mapping each omic name to a `(patients × features)` DataFrame
- `y_raw` — a Series of PAM50 subtype labels, indexed by patient ID

All views and the label are aligned on the same set of patients.

In [ ]:
# Path to the shared (workshop-wide) data directory
DATA_DIR = Path("../../data_tmp/TCGA-BRCA/")

# Load the three omic views we'll use throughout the workshop, plus the subtype labels
X_views, y_raw = load_omics(
    DATA_DIR,
    omic_keys=["transcriptomics", "proteomics", "methylation"],
)

## 3. Create a stratified train/test split

We split on **patient IDs only** (not on any single omic matrix) so the same patients
end up in the same split across every view. `stratify=y_raw` keeps the proportion of
each PAM50 subtype roughly equal between train and test — important here since some
subtypes (e.g. `Normal`, `Her2`) have relatively few patients.

`RANDOM_STATE` is fixed so the split is reproducible across the whole workshop.

In [ ]:
RANDOM_STATE = 42

train_ids, test_ids = train_test_split(
    y_raw.index,
    test_size=0.25,   # 75% train / 25% test
    random_state=RANDOM_STATE,
    stratify=y_raw,   # preserve subtype proportions in both splits
)

print(f"Training patients : {len(train_ids)}")
print(f"Test patients     : {len(test_ids)}")

## 4. Save the split for later notebooks

We persist the train/test patient IDs to a pickle file. Every subsequent notebook in
the workshop loads this file instead of re-splitting, so results stay consistent and
comparable across notebooks and participants.

In [ ]:
splits_path = DATA_DIR / "patient_splits.pkl"

with open(splits_path, "wb") as f:
    pickle.dump({"train_ids": train_ids, "test_ids": test_ids}, f)

print(f"Saved patient splits to: {splits_path}")